# Experiments 89
Testing para dataset color (RGB).

- **Model:**
    1. `yolov8m` *(Medium)*
- **Dataset:** 3.5m | 90º ***(v5i)***
    - Plus soil images
- **Crop Tiles:** 640px
- **Sizes:** small & mid
- **Experiments:**
    1. Evaluación del modelo para conjunto de testeo.

## Init

In [1]:
import os
import shutil
import fnmatch
import pickle

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 89.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

## Helper Functions

In [3]:
# Change for different file formats
reference = {
  "small": {
    "suffix": ".S",
    "file": "209"
  },
  "mid": {
    "suffix": ".M",
    "file": "503"
  },
  "large": {
    "suffix": ".L",
    "file": "000"
  }
}

In [4]:
# Clone config files
def copy_config(src_folder, dest_folder):
    """
    Copies files from src_folder to dest_folder, excluding subfolders.

    Args:
        src_folder: The path to the source folder.
        dest_folder: The path to the destination folder.
    """

    try:
        # Ensure destination folder exists
        os.makedirs(dest_folder, exist_ok=True)

        for filename in os.listdir(src_folder):
            src_path = os.path.join(src_folder, filename)
            dest_path = os.path.join(dest_folder, filename)

            if os.path.isfile(src_path):
                shutil.copy2(src_path, dest_path) #copy metadata as well.
                #Use shutil.copy for not copying metadata.
                print(f"Copied: {filename}")
            #else: #optional
                #print(f"Skipped (not a file): {filename}") #optional. Uncomment if you want to see the skipped folders.

        print("✅ Copying complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")


In [5]:
# Copy filtered dataset images/labels
def copy_and_filter_folder(src_folder, dest_folder, pattern):
    """
    Copies a folder and files that match the given pattern.
    Alerts the user when a folder or file already exists but *does not* overwrite.
    Creates only what is needed.

    :param src_folder: Path to the source folder.
    :param dest_folder: Path to the destination folder.
    :param pattern: Filename pattern to keep (e.g., "*.txt").
    """
    try:
        # Ensure destination folder exists
        if not os.path.exists(dest_folder):
            print(f"✓ Creating destination folder '{dest_folder}'.\n")
            os.makedirs(dest_folder)
        else:
            print(f"✓ Destination folder '{dest_folder}' already exists.\n")

        # Walk through the source folder
        for root, _, files in os.walk(src_folder):
            relative_path = os.path.relpath(root, src_folder)
            new_root = os.path.join(dest_folder, relative_path)

            if not os.path.exists(new_root):
                print(f"Creating subdirectory '{new_root}'")
                os.makedirs(new_root)
            else:
                print(f"❕Subdirectory '{new_root}' already exists.")
                print("Make sure the data inside is relevant. Otherwise, just delete the folder and repeat the cloning process.")

            for file in files:
                if fnmatch.fnmatch(file, pattern + "*"):
                    src_file = os.path.join(root, file)
                    dest_file = os.path.join(new_root, file)

                    if not os.path.exists(dest_file):
                        shutil.copy2(src_file, dest_file)  # copy metadata as well
                    else:
                        print(f"❗️File '{dest_file}' already exists. Skipping.")

            print(f" ✓ Copying files complete.\n")
        print("✅ Copying dataset complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")

In [6]:
def copy_directory(source, destination, overwrite=True):
    try:
        if overwrite and os.path.exists(destination):
            shutil.rmtree(destination)  # Remove existing directory
        shutil.copytree(source, destination)
        print(f"Directory '{source}' copied to '{destination}'")
        return True  # Indicate success
    except FileExistsError:
        print(f"Destination '{destination}' exists. Use overwrite=True to replace.")
        return False  # Indicate failure

In [7]:
# Load/Download prediction results (BB + confidence) as .pkl file

def save_results(results, filename):
    with open(filename, 'wb') as f:
        pickle.dump(results, f)

def load_results(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

In [8]:
def save_on_cloud(source: str, destination: str):
    """
    Saves a folder to a cloud storage location (e.g., Google Drive in Colab).

    Args:
        source (str): The path to the source folder.
        destination (str): The path to the destination folder (in cloud storage).
    """
    # 0. Input Validation (Assertions)
    assert isinstance(source, str), "Source must be a string."
    assert isinstance(destination, str), "Destination must be a string."

    try:
        # 1. Verify Source Folder
        if not os.path.exists(destination):
            os.makedirs(destination)

        # 2. Copy the Folder
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print("✅ Folder copied successfully:\n  ",source,"\n  -->",destination)

    except Exception as e:
        print(f"❌ An error occurred: {e}")

### Validation functions

In [9]:
cm = lambda results: results.confusion_matrix.matrix.tolist() if hasattr(results, 'confusion_matrix') and hasattr(results.confusion_matrix, 'matrix') else None

In [10]:
import json

def save_json(results):
  try:
    data_to_store = {
        "confusion_matrix": cm(results),
        "results_dict": results.results_dict,
        "speed": results.speed
    }

    # Convert the Python dictionary to a JSON string
    json_data = json.dumps(data_to_store, indent=4)

    # You can now save this JSON string to a file
    folder = str(results.save_dir)
    with open(f"/content/{folder}/results.json", "w") as f:
        f.write(json_data)

    print("✅ JSON file stored in:", folder)

  except Exception as e:
      print(f"❌ An error occurred: {e}")

  #return json_data


In [11]:
def gimme_metrics(results):
  matrix = cm(results)
  total_det = sum(sum(value) for value in matrix)
  percentages = []
  for row in matrix:
      values_percentages = []
      for value in row:
          if total_det != 0:
              percentage = (value / total_det) * 100
          else:
              percentage = 0.0
          values_percentages.append(f"{percentage:.2f}%")
      percentages.append(values_percentages)

  print("Total objects detected:", total_det)
  print("Confusion matrix:")
  for row in percentages:
      print(row)

  return matrix


In [12]:
def show_cm(TP, FP, FN):
    matrix = [[TP, FP], [FN, 0]]
    total_det = sum(sum(value) for value in matrix)
    percentages = []
    for row in matrix:
        values_percentages = []
        for value in row:
            if total_det != 0:
                percentage = (value / total_det) * 100
            else:
                percentage = 0.0
            values_percentages.append(f"{percentage:.2f}%")
        percentages.append(values_percentages)

    print("Total objects detected:", total_det)
    print("\nConfusion matrix:")
    for row in percentages:
        a, b = row
        print(f"[ {a} , {b} ]")

In [13]:
def show_metrics(TP, FP, FN):
    show_cm(TP, FP, FN)
    accuracy = TP/(TP+FP+FN)
    precision = TP/(TP+FP)
    recall = TP/(TP+FN)
    f1 = 2 * (precision * recall) / (precision + recall)
    f2 = 1.25 * (precision * recall) / (0.25 * precision + recall)
    fm = (precision * recall) ** 0.5
    print("\nMetrics:")
    print(f"- Accuracy: {accuracy:.3f}")
    print(f"- Precision: {precision:.3f}")
    print(f"- Recall: {recall:.3f}")
    print(f"- F1 Score: {f1:.3f}")
    print(f"- F½ Score: {f2:.3f}")
    print(f"- G-mean: {fm:.3f}")

# Datasets builder

## Importing from Drive

In [14]:
!rm -rf /content/sample_data

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
# Check if the cloud path is ok and the dataset can be found
!ls /content/drive/MyDrive/YOLO

 3.5m.v3i.yolov8.640px
 3.5m.v3i.yolov8.640px.aug.v1
 3.5m.v3i.yolov8.640px.aug.v1.soil_aug
 3.5m.v3i.yolov8.640px_clahe
 3.5m.v3i.yolov8.640px.soil_aug
 3.5m.v4i.yolov8.640px
 3.5m.v4i.yolov8.640px_209
 3.5m.v4i.yolov8_blended.640px
 3.5m.v5i.yolov8.640px-2steps.aug2
 3.5m.v5i.yolov8.640px.aug.v1
 3.5m.v5i.yolov8_blended.640px.aug.v1
'7 Validation_experiments_5_(exp_88).ipynb'
 Inference
 models
 optuna_yolov8_f1_study.db
 save_optuna


In [17]:
drive_path = '/content/drive/MyDrive/YOLO'
drive_datasets_paths = os.listdir(drive_path)
drive_datasets = len(drive_datasets_paths)
if (drive_datasets) > 1:
    print("There are %d dataset options:" % drive_datasets)
else:
    print("Theres is only 1 dataset:")
drive_datasets_paths

There are 16 dataset options:


['3.5m.v3i.yolov8.640px',
 'Inference',
 'models',
 '3.5m.v3i.yolov8.640px.aug.v1',
 'optuna_yolov8_f1_study.db',
 '3.5m.v3i.yolov8.640px.soil_aug',
 '3.5m.v3i.yolov8.640px.aug.v1.soil_aug',
 '3.5m.v3i.yolov8.640px_clahe',
 '3.5m.v4i.yolov8.640px',
 '3.5m.v4i.yolov8_blended.640px',
 '3.5m.v4i.yolov8.640px_209',
 '3.5m.v5i.yolov8.640px-2steps.aug2',
 'save_optuna',
 '7 Validation_experiments_5_(exp_88).ipynb',
 '3.5m.v5i.yolov8.640px.aug.v1',
 '3.5m.v5i.yolov8_blended.640px.aug.v1']

In [19]:
choose_dataset = 12 #10
index = choose_dataset - 1
dataset_name = os.listdir(drive_path)[index]
print("Chosen dataset:", dataset_name)

Chosen dataset: 3.5m.v5i.yolov8.640px-2steps.aug2


***Readme:***
*   **Option 1:** is desirable if you need to test many subset combinations in the same session (avoid downloading data twice from the cloud).
*   **Option 2:** is desired if you are goint to work just with one dataset  (avoid downloading unnecessary data from the cloud).
*   **Option 3:** is best if you're just going to test one subset combination  (avoid downloading any data from the cloud).



In [ ]:
# Option 2 (download just the needed dataset)
#cloud_path = f"{drive_path}/{dataset_name}/"
#local_path = f"/content/YOLO/"

In [ ]:
# Option 3 (download just what's needed)
!mkdir '/content/YOLO/'
for split in ['valid', 'test', 'small', 'large']:
  cloud_path = f"{drive_path}/{dataset_name}/{split}/"
  local_path = f"/content/YOLO/{dataset_name}/"
  !mkdir $local_path
  !cp -r $cloud_path $local_path
!cp -r $yaml_path $local_path

mkdir: cannot create directory ‘/content/YOLO/’: File exists
mkdir: cannot create directory ‘/content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/’: File exists
mkdir: cannot create directory ‘/content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/’: File exists
mkdir: cannot create directory ‘/content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/’: File exists


In [24]:
src_folder = f"/content/YOLO/{dataset_name}"
data = f"{src_folder}/data.yaml"
data

'/content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/data.yaml'

---

In [27]:
models_path = f'{drive_path}/models'
drive_models_path = os.listdir(models_path)
drive_models = len(drive_models_path)
if (drive_models) > 1:
    print("There are %d dataset options:" % drive_models)
else:
    print("Theres is only 1 dataset:")
drive_models_path

There are 5 dataset options:


['best_e26.pt', 'best_e68.pt', 'best_e50.pt', 'best_e86.pt', 'best_e79.pt']

In [29]:
choose_model = 4
index = choose_model - 1
model_name = os.listdir(models_path)[index]
print("Chosen model:", model_name)

Chosen model: best_e86.pt


In [30]:
# Option 2 (download just the dataset needed)
model_cloud_path = f"{models_path}/{model_name}"
model_local_path = f"/content/YOLO/"
!cp -r $model_cloud_path $model_local_path
model_weights = f"/content/YOLO/{model_name}"

In [31]:
import re
match = re.search(r"e(\d+)\.", model_name)

if match:
    model_num = match.group(1)
    model_num = f"e{model_num}"
    print(model_num)
else:
    print("No se encontró el número en el nombre del archivo.")

e86


# Experimentation

In [32]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### Optimization

In [33]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [34]:
# Reduce VRAM usage by reducing fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

### Info

In [35]:
!nvidia-smi

Fri May 16 20:03:50 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [36]:
!yolo version

8.3.137


-----
# Full dataset
### *YOLOv8 Mid | Testing*
Color images dataset

In [37]:
# Garbage collection
import gc
for i in range(20):
  torch.cuda.empty_cache()
  gc.collect()

In [ ]:
# Load currently trained YOLO model
model = YOLO(model_weights)

In [ ]:
# Best values found with Optuna
iou = 0.326691441
conf = 0.261453146

## Validation: *optimized iou/conf*

In [41]:
# Validate the model
results = model.val(data=data,
            split='test',
            batch=64,
            conf=conf, # best values found with optuna
            iou=iou, # best values found with optuna
            verbose=True,
            save_json=True)

Ultralytics 8.3.137 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)


100%|██████████| 755k/755k [00:00<00:00, 138MB/s]

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2029.5±514.3 MB/s, size: 94.3 KB)



val: Scanning /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/test/labels... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<00:00, 1238.17it/s]

val: New cache created: /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/test/labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.24s/it]


                   all        108       4036      0.715       0.44      0.563      0.264
Speed: 5.4ms preprocess, 21.7ms inference, 0.0ms loss, 11.0ms postprocess per image
Saving runs/detect/val/predictions.json...
Results saved to runs/detect/val


In [42]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val


In [43]:
save_json(results)

✅ JSON file stored in: runs/detect/val


In [44]:
matrix = gimme_metrics(results)

Total objects detected: 4587.0
Confusion matrix:
['42.08%', '12.01%']
['45.91%', '0.00%']


### Save results

In [45]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination=f'/content/drive/MyDrive/RGB_testing_({model_num})/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/RGB_testing_(e86)/


### Metrics

In [46]:
matrix

[[1930.0, 551.0], [2106.0, 0.0]]

In [47]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4587.0

Confusion matrix:
[ 42.08% , 12.01% ]
[ 45.91% , 0.00% ]

Metrics:
- Accuracy: 0.421
- Precision: 0.778
- Recall: 0.478
- F1 Score: 0.592
- F½ Score: 0.691
- G-mean: 0.610


## Validation: *default iou/conf*

In [ ]:
# Validate the model
results = model.val(data=data,
            split='test',
            batch=64,
            verbose=True,
            save_json=True)

Ultralytics 8.3.137 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1894.1±360.2 MB/s, size: 97.4 KB)


val: Scanning /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/test/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:07<00:00,  3.74s/it]


                   all        108       4036      0.595      0.506      0.526      0.217
Speed: 7.4ms preprocess, 25.0ms inference, 0.0ms loss, 4.6ms postprocess per image
Saving runs/detect/val/predictions.json...
Results saved to runs/detect/val


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 5022.0
Confusion matrix:
['41.72%', '19.63%']
['38.65%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination=f'/content/drive/MyDrive/RGB_testing_({model_num})/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/RGB_testing_(e86)/


### Metrics

In [ ]:
matrix

[[2095.0, 986.0], [1941.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 5022.0

Confusion matrix:
[ 41.72% , 19.63% ]
[ 38.65% , 0.00% ]

Metrics:
- Accuracy: 0.417
- Precision: 0.680
- Recall: 0.519
- F1 Score: 0.589
- F½ Score: 0.640
- G-mean: 0.594


-----
# Small plants
### *YOLOv8 Mid | Testing*
Color images dataset

## Validation: *optimized iou/conf*

In [ ]:
# Validate the model
results = model.val(data=data,
            split='test',
            batch=64,
            conf=conf, # best values found with optuna
            iou=iou, # best values found with optuna
            verbose=True,
            save_json=True)

Ultralytics 8.3.137 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1463.0±401.2 MB/s, size: 77.5 KB)


val: Scanning /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/test/labels... 49 images, 5 backgrounds, 0 corrupt: 100%|██████████| 54/54 [00:00<00:00, 2170.07it/s]

val: New cache created: /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/test/labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:02<00:00,  2.62s/it]


                   all         54       1353      0.652      0.541      0.563       0.27
Speed: 0.2ms preprocess, 23.2ms inference, 0.0ms loss, 2.3ms postprocess per image
Saving runs/detect/val2/predictions.json...
Results saved to runs/detect/val2


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val2


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val2


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 1679.0
Confusion matrix:
['47.41%', '19.42%']
['33.17%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination=f'/content/drive/MyDrive/RGB_testing_({model_num})/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/RGB_testing_(e86)/


### Metrics

In [ ]:
matrix

[[796.0, 326.0], [557.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 1679.0

Confusion matrix:
[ 47.41% , 19.42% ]
[ 33.17% , 0.00% ]

Metrics:
- Accuracy: 0.474
- Precision: 0.709
- Recall: 0.588
- F1 Score: 0.643
- F½ Score: 0.681
- G-mean: 0.646


## Validation: *default iou/conf*

In [ ]:
# Validate the model
results = model.val(data=data,
            split='test',
            batch=64,
            verbose=True,
            save_json=True)

Ultralytics 8.3.137 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1575.9±543.3 MB/s, size: 90.7 KB)


val: Scanning /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/test/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:07<00:00,  3.52s/it]


                   all        108       4036      0.595      0.506      0.526      0.217
Speed: 5.7ms preprocess, 25.1ms inference, 0.0ms loss, 6.7ms postprocess per image
Saving runs/detect/val2/predictions.json...
Results saved to runs/detect/val2


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val2


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val2


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 5022.0
Confusion matrix:
['41.72%', '19.63%']
['38.65%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination=f'/content/drive/MyDrive/RGB_testing_({model_num})/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/RGB_testing_(e86)/


### Metrics

In [ ]:
matrix

[[2095.0, 986.0], [1941.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 5022.0

Confusion matrix:
[ 41.72% , 19.63% ]
[ 38.65% , 0.00% ]

Metrics:
- Accuracy: 0.417
- Precision: 0.680
- Recall: 0.519
- F1 Score: 0.589
- F½ Score: 0.640
- G-mean: 0.594


-----
# Large plants
### *YOLOv8 Mid | Testing*
Color images dataset

## Validation: *optimized iou/conf*

In [ ]:
# Validate the model
results = model.val(data=data,
            split='test',
            batch=64,
            conf=conf, # best values found with optuna
            iou=iou, # best values found with optuna
            verbose=True,
            save_json=True)

Ultralytics 8.3.137 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1379.9±360.7 MB/s, size: 95.1 KB)


val: Scanning /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/test/labels... 46 images, 8 backgrounds, 0 corrupt: 100%|██████████| 54/54 [00:00<00:00, 2110.58it/s]

val: New cache created: /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/test/labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:02<00:00,  2.59s/it]


                   all         54       2109      0.587      0.378      0.445      0.207
Speed: 0.2ms preprocess, 22.8ms inference, 0.0ms loss, 1.3ms postprocess per image
Saving runs/detect/val3/predictions.json...
Results saved to runs/detect/val3


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val3


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val3


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 2604.0
Confusion matrix:
['33.18%', '19.01%']
['47.81%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination=f'/content/drive/MyDrive/RGB_testing_({model_num})/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/RGB_testing_(e86)/


### Metrics

In [ ]:
matrix

[[864.0, 495.0], [1245.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 2604.0

Confusion matrix:
[ 33.18% , 19.01% ]
[ 47.81% , 0.00% ]

Metrics:
- Accuracy: 0.332
- Precision: 0.636
- Recall: 0.410
- F1 Score: 0.498
- F½ Score: 0.573
- G-mean: 0.510


## Validation: *default iou/conf*

In [ ]:
# Validate the model
results = model.val(data=data,
            split='test',
            batch=64,
            verbose=True,
            save_json=True)

Ultralytics 8.3.137 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1688.5±520.1 MB/s, size: 97.6 KB)


val: Scanning /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/test/labels... 46 images, 8 backgrounds, 0 corrupt: 100%|██████████| 54/54 [00:00<00:00, 1350.62it/s]

val: New cache created: /content/YOLO/3.5m.v5i.yolov8.640px-2steps.aug2/test/labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:02<00:00,  2.96s/it]


                   all         54       2109      0.485      0.458      0.407      0.166
Speed: 0.2ms preprocess, 25.9ms inference, 0.0ms loss, 2.1ms postprocess per image
Saving runs/detect/val3/predictions.json...
Results saved to runs/detect/val3


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val3


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val3


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 2863.0
Confusion matrix:
['33.67%', '26.34%']
['39.99%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination=f'/content/drive/MyDrive/RGB_testing_({model_num})/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/RGB_testing_(e86)/


### Metrics

In [ ]:
matrix

[[964.0, 754.0], [1145.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 2863.0

Confusion matrix:
[ 33.67% , 26.34% ]
[ 39.99% , 0.00% ]

Metrics:
- Accuracy: 0.337
- Precision: 0.561
- Recall: 0.457
- F1 Score: 0.504
- F½ Score: 0.537
- G-mean: 0.506
